# Совпадения классов FPB и MyFoodRepo v2.1

Запускать перед подготовкой датасета. Сравниваются **138 классов FPB** и **323 класса MyFoodRepo v2.1**. По полным словарям найдено **23 одинаковых названия**, **10 вероятных синонимов** и близкие категории разной детализации.

Канонические названия и ID берутся из FPB. Совпадающие имена первоначально объединены; синонимы и спорные пары остаются отдельными до вашего решения. Решения от Food-101 здесь не используются. Сравниваются вырезки конкретных объектов обоих источников, только из исходных обучающих изображений.

In [ ]:
from pathlib import Path
import os
import sys
import json
from collections import Counter

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "scripts/dataset_pipeline.py").is_file()), None)
sys.path.insert(0, str(ROOT))
DATA = ROOT / "data"
DATA.mkdir(exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(DATA / "cache/matplotlib"))

import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display
from scripts import dataset_pipeline as dp
from scripts import myfoodrepo as mfr

cats = dp.catalogs(DATA)
display(pd.DataFrame([{"source": source, "classes": len(info["names"]), "revision": info["revision"]}
                      for source, info in cats.items()]))

## 1. Таблица соответствий

`merge` вместе с `reviewed=True` объединяет класс MyFoodRepo с указанным классом FPB. `keep_separate` сохраняет его отдельно. Один класс MyFoodRepo может иметь только одну целевую категорию; несколько его классов можно объединить в один FPB.

Таблица находится в `data/review/fpb_myfoodrepo_class_decisions.csv`. Повторный запуск сохраняет принятые решения.

In [ ]:
DECISIONS_PATH = DATA / "review/fpb_myfoodrepo_class_decisions.csv"
review = pd.read_csv(DECISIONS_PATH) if DECISIONS_PATH.exists() else dp.review_table(cats)
dp.validate_decisions(review, cats)
if not DECISIONS_PATH.exists():
    dp.save_decisions(review, DECISIONS_PATH, cats)
display(review)
display(review.groupby("kind").size().rename("pairs").to_frame())

## 2. Загрузка примеров

Кешируются выбранные изображения и список примеров. FPB ищется в пределах `MAX_SCAN` файлов разметки; редкие классы могут быть представлены недостаточно. Для MyFoodRepo подбираются изображения с нужными объектами, по возможности несколько на класс. Одно изображение может покрывать несколько классов.

Поля `bbox` зеркала MyFoodRepo содержат несогласованные координаты. Рамки вычисляются по полигонам. Вырожденные компоненты из одной-двух точек игнорируются, точные повторы объекта того же класса на одном фото удаляются. Подробности записываются в `data/metadata/myfoodrepo_v2_1/import_audit.json`. Масса не используется.

In [ ]:
SAMPLES_PER_CLASS = 2
MAX_SCAN = 1600
WORKERS = 8
SHOW_EXACT_NAMES = False  # True — показать также все 23 пары с одинаковыми названиями.

records, coverage = dp.preview_samples(DATA, cats, per_class=SAMPLES_PER_CLASS,
                                       max_scan=MAX_SCAN, workers=WORKERS)
counts = Counter()
for row in records:
    if row["source_split"] == "train":
        counts.update((row["source"], obj["source_class_id"]) for obj in row["objects"] or [])
fpb_ids = {name: i for i, name in enumerate(cats["fpb"]["names"])}
mfr_ids = dict(zip(cats["myfoodrepo"]["names"], cats["myfoodrepo"]["category_ids"]))
display(pd.DataFrame([
    {"fpb_class": row.fpb_class, "myfoodrepo_class": row.myfoodrepo_class,
     "fpb_objects": counts[("fpb", fpb_ids[row.fpb_class])],
     "myfoodrepo_objects": counts[("myfoodrepo", mfr_ids[row.myfoodrepo_class])]}
    for row in review.itertuples(index=False)
]))
print("Фото в preview:", len(records))
print("Недостаточное покрытие FPB:", coverage["missing"])

## 3. Визуальное сравнение

По умолчанию показаны вероятные синонимы и спорные пары. Оцените состав, приготовление и ширину категории: например, `tea` шире `black_tea`, а `pizza_margherita_baked` уже `pizza`. Пустая ячейка означает отсутствие примера, а не подтверждение совпадения. PNG-отчёты не сохраняются.

In [ ]:
pairs = review if SHOW_EXACT_NAMES else review[review.kind != "exact_name"]
for pair in pairs.itertuples(index=False):
    print(f"{pair.kind}: {pair.fpb_class} / {pair.myfoodrepo_class} — {pair.note}")
    fig = dp.show_class_pair(records, cats, pair.fpb_class, pair.myfoodrepo_class,
                             samples=SAMPLES_PER_CLASS)
    plt.close(fig)

## 4. Решения после просмотра

In [ ]:
DECISIONS = {
    source_name: {"decision": "merge", "reviewed": True,
                  "note": "Объединено по решению после просмотра"}
    for source_name in review["myfoodrepo_class"]
}
DECISIONS["pancake"] = {"decision": "keep_separate", "reviewed": True,
                        "note": "pancake и bliny — разные блюда"}
for source_name, decision in DECISIONS.items():
    mask = review.myfoodrepo_class == source_name
    if mask.sum() != 1:
        raise ValueError(f"Нет единственного соответствия для {source_name}")
    for column in ("fpb_class", "decision", "reviewed", "note"):
        if column in decision:
            review.loc[mask, column] = decision[column]
dp.save_decisions(review, DECISIONS_PATH, cats)
display(review[["fpb_class", "myfoodrepo_class", "decision", "reviewed", "note"]])
names, mapping, mapping_rows = dp.class_mapping(cats, review)
print("Классов в объединённом словаре:", len(names))
display(pd.DataFrame(mapping_rows).query("rule == 'reviewed_alias'"))

Далее запустите `02_prepare_dataset.ipynb`. Он прочитает эту таблицу и применит единый словарь к обеим обучающим выборкам. Food-101 остаётся отдельным набором фотографий без разметки для качественной проверки итоговой модели.